# vector-normalize-keepdim composite — cx13: row-wise L2 normalize via keepdim broadcast

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `broadcasting-rules`, `vector-normalize-keepdim`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "vector-normalize-keepdim"
DD_ATOM_IDS = ["broadcasting-rules", "vector-normalize-keepdim"]
DD_SUBTOPICS = ["Numpy: Vectorization and broadcasting", "PyTorch: vector normalize keepdim"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Row normalize = keepdim + broadcast — two atoms in one expression

1. **`broadcasting-rules`** — right-align shapes, left-pad with 1s. `(N, D)`
   divided by `(N,)` tries to align the `N` axis with the trailing axis of
   the matrix (size `D`). Mismatch → either an error or, worse, a silent
   wrong-axis broadcast.
2. **`vector-normalize-keepdim`** — `x.norm(dim=-1, keepdim=True)` returns
   shape `(N, 1)` instead of `(N,)`. The trailing 1 is what makes the divide
   broadcast cleanly across the `D` axis.

Composition: `keepdim=True` is *literally the bridge* that makes the norm
tensor broadcast-compatible. Drop the keepdim and broadcasting either errors
or aligns to the wrong axis — either way you don't get row normalization.


### Composite Exercise — row-wise L2 normalize via keepdim broadcast

**Atoms exercised together**: `broadcasting-rules`, `vector-normalize-keepdim`

Implement `cx13_row_normalize(x)`. Given a batch shaped `(B, D)`:

1. Compute the per-row L2 norm with `keepdim=True` → shape `(B, 1)`.
2. Divide `x` by that norm; broadcasting expands `(B, 1)` over `D`.
3. Return the normalized tensor, same shape as `x`.

Must use `keepdim=True`. The test verifies the norm tensor was kept as
`(B, 1)` (not squeezed) by comparing intermediate shapes via a helper.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx13_row_normalize(x: Tensor) -> Tensor:
    """L2-normalize each row of x with keepdim broadcasting."""
    raise NotImplementedError()


def _test_cx13():
    # --- canonical 3-4-5 row ---
    x = t.tensor([[3.0, 4.0], [0.0, 1.0], [-1.0, 0.0]])
    out = cx13_row_normalize(x)
    assert out.shape == x.shape
    expected = t.tensor([[0.6, 0.8], [0.0, 1.0], [-1.0, 0.0]])
    assert t.allclose(out, expected, atol=1e-6), f'got {out}'

    # --- random batch: every row has unit norm ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(8, 16, generator=rng) + 0.1
    Y = cx13_row_normalize(X)
    assert Y.shape == X.shape
    row_norms = Y.norm(dim=-1)
    assert t.allclose(row_norms, t.ones(8), atol=1e-5), f'row norms: {row_norms}'

    # --- keepdim was used: broadcasting must succeed on (B, D) / (B, 1) ---
    # We verify by checking that a non-square D works (would error if norm was (B,))
    X2 = t.randn(3, 5, generator=rng) + 0.5
    Y2 = cx13_row_normalize(X2)
    assert Y2.shape == X2.shape
    assert t.allclose(Y2.norm(dim=-1), t.ones(3), atol=1e-5)

    # --- value witness: each row equals x_row / ||x_row|| ---
    for i in range(X.shape[0]):
        assert t.allclose(Y[i], X[i] / X[i].norm(), atol=1e-5)

    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
def cx13_row_normalize(x: Tensor) -> Tensor:
    # keepdim=True keeps the divisor as (B, 1) so broadcasting aligns
    # the trailing 1 against the D axis and expands across it.
    return x / x.norm(dim=-1, keepdim=True)

```

The `keepdim=True` IS the broadcasting-rules atom in action: it produces a
`(B, 1)` divisor whose trailing 1 right-aligns against the `D` axis of `x`,
so the divide broadcasts cleanly. Without it the divisor is `(B,)`, which
right-aligns against `D` — wrong axis.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["Numpy: Vectorization and broadcasting", "PyTorch: vector normalize keepdim"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()